# 経費申請分析

このノートブックでは、プラグインを使用してローカルの領収書画像から出張経費を処理し、経費申請メールを生成し、円グラフを使って経費データを可視化するエージェントを作成する方法を示します。エージェントはタスクの状況に応じて動的に関数を選択します。

手順：
1. OCRエージェントがローカルの領収書画像を処理し、出張経費データを抽出します。
2. メールエージェントが経費申請メールを生成します。

### 出張経費のシナリオ例：
あなたが別の都市でのビジネスミーティングのために出張している社員であると想像してください。会社にはすべての妥当な出張関連経費を払い戻す方針があります。以下は出張経費の内訳です：
- 交通費：
自宅の都市から目的地の都市への往復航空券。
空港までのタクシーまたはライドシェアサービス。
目的地の都市でのローカル交通（公共交通機関、レンタカー、タクシーなど）。

- 宿泊費：
会議会場近くの中級ビジネスホテルに３泊の宿泊。

- 食事：
会社の日当ポリシーに基づく朝食、昼食、夕食の毎日の食事手当。

- 雑費：
空港での駐車料金。
ホテルでのインターネット接続料金。
チップや小さなサービス料。

- 書類：
すべての領収書（航空券、タクシー、ホテル、食事など）および払い戻し用の完成した経費精算書を提出します。


## 必要なライブラリをインポートする

ノートブックに必要なライブラリとモジュールをインポートします。


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 支出モデルの定義

個々の支出用にPydanticモデルを作成し、ユーザーのクエリを構造化された支出データに変換するExpenseFormatterクラスを作成します。

各支出は次の形式で表されます：
`{'date': '07-Mar-2025', 'description': '目的地へのフライト', 'amount': 675.99, 'category': '交通費'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## ツールの定義 - メールの生成

経費申請を送信するためのメールを生成するツール関数を作成します。
- このツールは Microsoft Agent Framework の `@tool` デコレーターを使用します。
- 経費の合計金額を計算し、詳細をメール本文の形式に整えます。


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# レシート画像から旅費を抽出するツール

レシート画像から旅費を抽出するツール関数を作成します。
- このツールは Microsoft Agent Framework の `@tool` デコレーターを使用します。
- レシート画像を読み込み、base64 エンコードして、エージェントが解析できるようにデータ URI を返します。


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## 経費処理

エージェントを定義し、`WorkflowBuilder` を使用して順次ワークフローに接続します。
- OCRエージェントは、`load_receipt_image` ツールを使用してレシート画像から構造化された経費データを抽出します。
- Emailエージェントは、抽出されたデータを受け取り、`generate_expense_email` ツールを使用して専門的な経費請求メールを作成します。
- `WorkflowBuilder` の `add_edge` を使って順次パイプラインを作成します：OCRエージェント → Emailエージェント。


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## メイン関数

連続したワークフローを構築し、実行して領収書画像を処理し、経費請求メールを生成します。


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を使用して翻訳されました。正確性の向上に努めておりますが、自動翻訳には誤りや不正確な箇所が含まれる場合があります。原文の言語で記載された書類が正式な情報源であることをご理解ください。重要な情報については、専門の人間による翻訳をご利用いただくことをお勧めします。本翻訳の利用により生じた誤解や誤訳について、当方は一切の責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
